# Antagonistic pleiotropy

Ontogenetic matrix entries in {-1, 0, 1}, no environmental noise (`?EvoMod.Antagonistic`).
Compares a *corridor* regime (only module 1 is selected) with *divergent* selection
(module 1 up, module 2 down), and tracks how effect vectors are classified
(Modular, Antagonistic, Local, Integrated, Neutral).

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using EvoMod, EvoMod.Antagonistic
using Gadfly, DataFrames, LinearAlgebra, Random, Statistics
include("plots.jl")
Random.seed!(42)

In [ ]:
# Parameters
n_e  = 5000
m    = 100
p    = 4
mu   = 0.0005
mu_b = 0.0001
mu_σ = 0.02

omega_var = 2
delta_theta_speed = 40/1000

regimes     = ["Drift", "Stab", "Direct", "Stab"]
generations = [2000, 2000, 2000, 1000]
thin        = 1
plot_start  = 2000   # first recorded moment shown in the plots

random_ind = Individual(m, p, mu, mu_b, mu_σ)

delta_theta   = delta_theta_speed * [ones(p ÷ 2); -ones(p ÷ 2)]    # divergent
delta_theta_1 = delta_theta_speed * [ones(p ÷ 2); zeros(p ÷ 2)]    # corridor, module 1
delta_theta_2 = delta_theta_speed * [zeros(p ÷ 2); -ones(p ÷ 2)]   # corridor, module 2
omega = twoModuleMatrix(omega_var * ones(p), 0.8, 0.8)

In [ ]:
corridor_pop = run_pop(random_ind, n_e, regimes, generations;
                       delta_theta = delta_theta_1, omega = omega, thin = thin)
divergent_pop = run_pop(random_ind, n_e, regimes, generations;
                        delta_theta = delta_theta, omega = omega, thin = thin)

In [ ]:
all_plots(corridor_pop, "corridor.pdf"; start = plot_start)
all_plots(divergent_pop, "divergent.pdf"; start = plot_start)

In [ ]:
plotCorrelations(divergent_pop, plot_start)

In [ ]:
plotTraits(divergent_pop, plot_start)

In [ ]:
plotPleiotropy(divergent_pop, plot_start)

In [ ]:
divergent_pop.moments[end]["corrG"]

Effect vectors classified as Modular in the population-mean `b`:

In [ ]:
b = round.(Int, divergent_pop.moments[end]["mean_b"])
b[:, classifyBj(b) .== "Modular"]